低波动、ETF与银行归因复算

2021-09-01—2025-08-06，资金20万元，万五佣金、每笔最低5元。原规则不变，封存收益未使用。

In [1]:
from pathlib import Path
import json, hashlib
import pandas as pd
import numpy as np
root=Path('/Users/guo/Documents/stocks/quant-research')
run=root/'artifacts/lowvol-benchmark-20261013-v1'
audit=json.loads((run/'etf-independent-audit.json').read_text())
for path,digest in audit['source_hashes'].items():
    assert hashlib.sha256((root/path).read_bytes()).hexdigest()==digest,path
print({'ETF_accounts':audit['accounts'],'max_cash_NAV_error':audit['max_cash_nav_error'],'sealed_return_rows_used':audit['sealed_return_rows_used']})
print(pd.read_csv(run/'main-comparison.csv')[['direction','net_return','cagr','max_drawdown','mean_exposure']].to_string(index=False))


{'ETF_accounts': 78, 'max_cash_NAV_error': 0.0, 'sealed_return_rows_used': 0}
        direction  net_return      cagr  max_drawdown  mean_exposure
           原低波动选股    0.563471  0.125721      0.129098       0.889626
 沪深300ETF（510300）   -0.073107 -0.019916      0.308827       0.851659
 中证500ETF（510500）   -0.065555 -0.017806      0.356275       0.873914
    银行ETF（512800）    0.486110  0.110685      0.209780       0.904516
  红利低波ETF（512890）    0.385917  0.090330      0.145981       0.902822
45%银行＋45%沪深300＋现金    0.184043  0.045782      0.228517       0.899452
       排除银行后重跑低波动    0.364137  0.085764      0.148099       0.883856


In [2]:
parts=pd.read_csv(run/'attribution-daily-base.csv')
profit=parts[['bank_pnl','nonbank_pnl']].sum()
old=pd.read_parquet(root/'artifacts/strategy-screen-20261013-v1/results/base/lowvol60/nav.parquet')
assert abs(profit.sum()-(old.equity.iloc[-1]-200000))<1e-6
print(profit.to_string())
print('bank_share_of_total_net_profit',profit.bank_pnl/profit.sum())
decision=json.loads((run/'decision.json').read_text())
phase=pd.DataFrame(decision['phase_comparisons'])
print(phase.to_string(index=False))
assert int(phase.both_beaten.sum())==decision['actual_phase_wins']
print('recommend_entering_sealed_test',decision['recommend_entering_sealed_test'])


bank_pnl       83824.048878
nonbank_pnl    28870.162450
bank_share_of_total_net_profit 0.7438185856260445
scenario  stock_net_return  etf_net_return  stock_drawdown  etf_drawdown  stock_calmar  etf_calmar  both_beaten  nonbank_net_return
    base          0.563471        0.385917        0.129098      0.145981      0.973841    0.618783         True            0.364137
  phase5          0.327147        0.355761        0.152564      0.135935      0.510493    0.617891        False            0.080561
 phase10          0.129432        0.374823        0.222371      0.126914      0.147403    0.693468        False           -0.036525
 phase15          0.092028        0.405067        0.217813      0.127126      0.108361    0.741805        False           -0.133192
recommend_entering_sealed_test False


In [3]:
for path in (run/'etfs').glob('*/*/*/nav.parquet'):
    nav=pd.read_parquet(path)
    v=np.r_[200000.,nav.equity]
    scenario,code,rule=path.parent.relative_to(run/'etfs').parts
    rows=pd.read_csv(run/'etf-summary.csv')
    item=rows[(rows.scenario==scenario)&(rows.code==code)&(rows.rule==rule)].iloc[0]
    assert abs(v[-1]/200000.-1-item.net_return)<1e-10
    assert abs(-(v/np.maximum.accumulate(v)-1).min()-item.max_drawdown)<1e-10
print('All 78 ETF net returns and daily drawdowns checked.')


All 78 ETF net returns and daily drawdowns checked.


银行利润占比是持仓会计贡献，不是行业被动行情的因果解释。回归为描述性分析。全部结论及输入假设见同名报告。当前版本1/4起点通过，未达到预先要求的3/4，因此不进入封存期。